# LiDAR Gap Analysis — Puerto Rico G-LiHT
**Luquillo Experimental Forest** | Gap creation: March 2017 (pre-Maria) → May 2018 (post-Maria)

Pipeline:
1. Load & align LAS files (2017 and 2018 only)
2. Compute absolute C2C nearest-neighbour distances
3. Extract connected components to isolate damage clumps
4. Power-law gap size distribution analysis

In [ ]:
import os
import re
import csv
import sys
import time
import gzip
import signal
import shutil
import tempfile
from math import log2, sqrt
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

import cloudComPy as cc
cc.initCC()
print('[INIT] CloudComPy initialized')

## 1. Configuration

In [ ]:
# ── Input directories ────────────────────────────────────────────────────────
DIR_2017 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_11March2017_FIA12/lidar/las')
DIR_2018 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_2May018_FIA12/lidar/las')

# ── Output directories ───────────────────────────────────────────────────────
OUT_DIR  = Path('/home/ogh6/Downloads/raw_c2c_output')
CSV_DIR  = OUT_DIR / 'csv'
PLOT_DIR = OUT_DIR / 'plots'
BIN_DIR  = OUT_DIR / 'bin'
for d in [OUT_DIR, CSV_DIR, PLOT_DIR, BIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Processing parameters ────────────────────────────────────────────────────
USE_SUBSAMPLE    = True
SUBSAMPLE_N      = 5_000_000
THREADS          = 12
GRID_TIMEOUT_SEC = 600        # skip grid if it takes longer than 10 minutes

# ── C2C analysis parameters ──────────────────────────────────────────────────
NOISE_CEIL_M   = 20.0
C2C_THRESHOLDS = [2, 5, 10, 15]
EXTREME_PCTILE = 95
HIST_MAX_M     = 20.0
HIST_BINS      = 200

# ── Connected component parameters ───────────────────────────────────────────
CC_CONFIG = {
    'damage_threshold':     2.0,
    'min_component_pts':    100,
    'min_gap_area_m2':      10.0,
    'overlap_min_fraction': 0.80,
    'overlap_buffer_m':     50.0,
    'max_components':       100_000,
}

GRID_RE  = re.compile(r'(l\d+s\d+)', re.IGNORECASE)
_TMP_DIR = Path(tempfile.gettempdir()) / f'cc_c2c_{int(time.time())}'
_TMP_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  2017 : {DIR_2017}\n  2018 : {DIR_2018}\n  Out  : {OUT_DIR}')
print(f'  Grid timeout: {GRID_TIMEOUT_SEC}s')

## 2. Timeout Helper
Wraps any block of code with a hard timeout using `SIGALRM` (Linux only). If a grid hangs, it raises `GridTimeoutError` and the main loop skips to the next file.

In [ ]:
class GridTimeoutError(Exception):
    pass

def _timeout_handler(signum, frame):
    raise GridTimeoutError('Grid processing timed out')

signal.signal(signal.SIGALRM, _timeout_handler)

def start_timeout(seconds):
    signal.alarm(int(seconds))

def cancel_timeout():
    signal.alarm(0)

print(f'Timeout handler ready — grids will be skipped after {GRID_TIMEOUT_SEC}s.')

## 3. Utility Functions

In [ ]:
def tnow(): return time.time()
def fmt_s(dt):
    if dt < 60:   return f'{dt:.1f}s'
    if dt < 3600: return f'{dt/60:.1f}m'
    return f'{dt/3600:.2f}h'

def mem_gb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    except ImportError:
        return -1.0

def gunzip_if_needed(p):
    p = Path(p)
    if p.suffix != '.gz': return p
    out = _TMP_DIR / p.name[:-3]
    if out.exists() and out.stat().st_size > 0: return out
    print(f'  [UNZIP] {p.name} ...', flush=True)
    t0 = tnow()
    with gzip.open(p, 'rb') as fin, open(out, 'wb') as fout:
        shutil.copyfileobj(fin, fout)
    print(f'  [UNZIP] done {fmt_s(tnow()-t0)}', flush=True)
    return out

def grid_id_from_name(p):
    m = GRID_RE.search(Path(p).name)
    if not m: raise ValueError(f'No grid ID in {p}')
    return m.group(1).lower()

def find_match(folder, grid):
    for f in Path(folder).glob('*.las*'):
        m = GRID_RE.search(f.name)
        if m and m.group(1).lower() == grid: return f
    return None

def load_cloud(p, shift=None):
    t0 = tnow()
    print(f'  [LOAD] {Path(p).name} ...', flush=True)
    if shift is not None:
        cloud = cc.loadPointCloud(filename=str(p), mode=cc.CC_SHIFT_MODE.XYZ,
                                  x=shift[0], y=shift[1], z=shift[2])
    else:
        cloud = cc.loadPointCloud(str(p))
    if cloud is None: raise RuntimeError(f'Failed to load {p}')
    applied_shift = (0.0, 0.0, 0.0)
    if cloud.isShifted():
        gs = cloud.getGlobalShift()
        applied_shift = (float(gs[0]), float(gs[1]), float(gs[2]))
    print(f'  [LOAD] done {fmt_s(tnow()-t0)} | pts={cloud.size():,} | mem={mem_gb():.2f}GB', flush=True)
    return cloud, applied_shift

def safe_delete(entity):
    try:
        if entity is not None: cc.deleteEntity(entity)
    except Exception:
        pass

def write_csv(filepath, rows, fieldnames=None):
    if not rows: return
    if fieldnames is None:
        seen = {}
        for r in rows:
            for k in r: seen[k] = True
        fieldnames = list(seen)
    with open(filepath, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
    print(f'  [CSV] {filepath}', flush=True)

print('Utility functions ready.')

## 4. Normalisation, Subsampling & C2C

In [ ]:
def check_normalization(cloud, label=''):
    bb = cloud.getOwnBB()
    z_min, z_max = float(bb.minCorner()[2]), float(bb.maxCorner()[2])
    sf_names  = list(cloud.getScalarFieldDic().keys())
    has_class = any('classif' in n.lower() or 'class' in n.lower() for n in sf_names)
    if z_min > 50:        normalised, note = False, f'Z min={z_min:.1f}m — raw GPS. C2C still valid.'
    elif -2 <= z_min < 5: normalised, note = True,  'Appears height-normalised.'
    else:                 normalised, note = None,  f'Ambiguous (z_min={z_min:.1f}). Check manually.'
    return {'label': label, 'z_min': z_min, 'z_max': z_max,
            'has_classification': has_class, 'likely_normalised': normalised,
            'note': note, 'scalar_fields': ', '.join(sf_names)}

def print_norm_report(info):
    status = {True: 'YES', False: 'NO', None: 'UNCLEAR'}[info['likely_normalised']]
    print(f"  [NORM] {info['label']}: z_min={info['z_min']:.1f} z_max={info['z_max']:.1f} | normalised={status}")

def subsample_cloud(cloud, target_n):
    if cloud.size() <= target_n: return cloud
    t0 = tnow()
    print(f'  [SUB] {cloud.size():,} -> ~{target_n:,} ...', flush=True)
    ref = cc.CloudSamplingTools.subsampleCloudWithOctree(
        cloud, int(target_n), cc.SUBSAMPLING_CELL_METHOD.NEAREST_POINT_TO_CELL_CENTER)
    sub, res = cloud.partialClone(ref)
    if res != 0: raise RuntimeError(f'partialClone failed (code {res})')
    print(f'  [SUB] done {fmt_s(tnow()-t0)} | out={sub.size():,} | mem={mem_gb():.2f}GB', flush=True)
    return sub

def compute_c2c(src, tgt, sf_name, threads=12):
    t0 = tnow()
    print(f'  [C2C] {sf_name}: approx pass ...', flush=True)
    approx_stats = cc.DistanceComputationTools.computeApproxCloud2CloudDistance(src, tgt)
    if approx_stats:
        print(f'  [C2C] approx: min={approx_stats[0]:.2f} max={approx_stats[1]:.2f} mean={approx_stats[2]:.2f}', flush=True)
    best_level = cc.DistanceComputationTools.determineBestOctreeLevel(src, None, tgt)
    print(f'  [C2C] octree level = {best_level}', flush=True)
    params = cc.Cloud2CloudDistancesComputationParams()
    params.octreeLevel = best_level; params.multiThread = True; params.maxThreadCount = int(threads)
    ret = cc.DistanceComputationTools.computeCloud2CloudDistances(src, tgt, params)
    if ret < 0: print(f'  [C2C] WARNING: returned {ret}', flush=True)
    sf = src.getScalarField(src.getNumberOfScalarFields() - 1)
    sf.setName(sf_name); sf.computeMinAndMax()
    for name, idx in src.getScalarFieldDic().items():
        if 'approx' in name.lower():
            src.deleteScalarField(idx); break
    print(f'  [C2C] {sf_name} done in {fmt_s(tnow()-t0)}', flush=True)
    return sf_name

def get_sf_as_array(cloud, sf_name):
    sf = cloud.getScalarField(sf_name)
    if sf is None:
        d = cloud.getScalarFieldDic()
        if sf_name in d: sf = cloud.getScalarField(d[sf_name])
    if sf is None:
        raise KeyError(f"SF '{sf_name}' not found. Available: {list(cloud.getScalarFieldDic().keys())}")
    return np.asarray(sf.toNpArrayCopy(), dtype=np.float64)

print('Normalisation, subsampling, C2C functions ready.')

## 5. C2C Statistical Analysis & Plotting

In [ ]:
def c2c_summary_stats(c2c, label=''):
    clean = c2c[c2c <= NOISE_CEIL_M]; n_total = len(c2c); n_clean = len(clean)
    out = {'label': label, 'n_total': n_total, 'n_clean': n_clean,
           'n_noise': n_total-n_clean, 'pct_noise': (n_total-n_clean)/max(n_total,1)*100}
    if n_clean > 0:
        out.update({'mean': float(np.mean(clean)),   'median': float(np.median(clean)),
                    'std':  float(np.std(clean)),    'min':    float(np.min(clean)),
                    'max':  float(np.max(clean)),    'p05':    float(np.percentile(clean,5)),
                    'p25':  float(np.percentile(clean,25)), 'p75': float(np.percentile(clean,75)),
                    'p95':  float(np.percentile(clean,95)), 'p99': float(np.percentile(clean,99))})
    return out

def c2c_threshold_fractions(c2c, thresholds=C2C_THRESHOLDS):
    clean = c2c[c2c <= NOISE_CEIL_M]; n = max(len(clean), 1)
    return {t: float(np.sum(clean >= t)/n) for t in thresholds}

def extreme_value_report(c2c, label='', percentile=EXTREME_PCTILE):
    clean = c2c[c2c <= NOISE_CEIL_M]
    if len(clean) == 0: return {'label': label, 'error': 'no clean points'}
    thresh = float(np.percentile(clean, percentile)); extreme = clean[clean > thresh]
    return {'label': label, 'percentile': percentile, 'threshold_m': thresh,
            'n_extreme': len(extreme), 'n_clean': len(clean),
            'extreme_mean': float(np.mean(extreme)),   'extreme_median': float(np.median(extreme)),
            'extreme_std':  float(np.std(extreme)),    'extreme_max':    float(np.max(extreme)),
            'pct_2_5m':   float(np.sum((extreme>=2) &(extreme<5)) /max(len(extreme),1)*100),
            'pct_5_10m':  float(np.sum((extreme>=5) &(extreme<10))/max(len(extreme),1)*100),
            'pct_10_15m': float(np.sum((extreme>=10)&(extreme<15))/max(len(extreme),1)*100),
            'pct_15_20m': float(np.sum((extreme>=15)&(extreme<20))/max(len(extreme),1)*100)}

def plot_grid(grid_id, c2c_arr, out_dir=PLOT_DIR):
    clean = c2c_arr[c2c_arr <= NOISE_CEIL_M]
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Grid {grid_id.upper()} — C2C Gap Analysis (2017→2018)', fontsize=14, fontweight='bold')
    edges = np.linspace(0, HIST_MAX_M, HIST_BINS+1)

    ax=axes[0,0]; ax.hist(clean, bins=edges, density=True, color='red', alpha=0.7)
    ax.set_xlabel('C2C (m)'); ax.set_ylabel('Density'); ax.set_title('(a) C2C Distribution'); ax.set_xlim(0, HIST_MAX_M)

    ax=axes[0,1]; s=np.sort(clean); cdf=np.arange(1,len(s)+1)/len(s); step=max(len(s)//10000,1)
    ax.plot(s[::step], cdf[::step], color='red', lw=1.5)
    ax.set_xlabel('C2C (m)'); ax.set_ylabel('CDF'); ax.set_title('(b) CDF'); ax.set_xlim(0, HIST_MAX_M); ax.grid(True, alpha=0.3)

    ax=axes[0,2]; x_pos=np.arange(len(C2C_THRESHOLDS)); tf=c2c_threshold_fractions(c2c_arr)
    ax.bar(x_pos, [tf[t]*100 for t in C2C_THRESHOLDS], color='red', alpha=0.7)
    ax.set_xticks(x_pos); ax.set_xticklabels([f'>={t}m' for t in C2C_THRESHOLDS])
    ax.set_ylabel('% clean pts'); ax.set_title('(c) Threshold Exceedance')

    ax=axes[1,0]; ev=extreme_value_report(c2c_arr); bands=['2-5m','5-10m','10-15m','15-20m']
    keys=['pct_2_5m','pct_5_10m','pct_10_15m','pct_15_20m']
    ax.bar(np.arange(4), [ev.get(k,0) for k in keys], color='red', alpha=0.7)
    ax.set_xticks(np.arange(4)); ax.set_xticklabels(bands)
    ax.set_ylabel(f'% extreme (>{EXTREME_PCTILE}th pctile)'); ax.set_title('(d) Extreme Value Bands')

    ax=axes[1,1]; fine=np.linspace(0, 5, 100)
    ax.hist(clean, bins=fine, density=True, color='red', alpha=0.7)
    ax.set_xlabel('C2C (m)'); ax.set_ylabel('Density'); ax.set_title('(e) Detail 0–5m')
    ax.axvline(2.0, color='gray', linestyle='--', alpha=0.5, label='2m threshold')
    ax.legend(fontsize=8)

    ax=axes[1,2]; ax.axis('off')
    s1 = c2c_summary_stats(c2c_arr)
    txt = (f"2017→2018 (Hurricane Maria)\n"
           f"  n={s1['n_clean']:,}  noise={s1['n_noise']:,} ({s1['pct_noise']:.1f}%)\n"
           f"  mean={s1.get('mean',0):.2f}  med={s1.get('median',0):.2f}  std={s1.get('std',0):.2f}\n"
           f"  p95={s1.get('p95',0):.2f}  p99={s1.get('p99',0):.2f}\n\nNOISE CEIL: {NOISE_CEIL_M}m")
    ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=9, va='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.set_title('(f) Summary')
    plt.tight_layout(rect=[0,0,1,0.95])
    out = Path(out_dir)/f'{grid_id}_c2c_analysis.png'
    fig.savefig(str(out), dpi=150); plt.close(fig)
    print(f'  [PLOT] {out.name}', flush=True)

print('C2C analysis and plotting functions ready.')

## 6. Connected Component Functions

In [ ]:
def _cfg(config, key):
    return config[key] if config and key in config else CC_CONFIG[key]

def _bb_xy(cloud):
    bb=cloud.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
    return (lo[0], lo[1], hi[0], hi[1])

def _intersect_xy(boxes):
    xmin=max(b[0] for b in boxes); ymin=max(b[1] for b in boxes)
    xmax=min(b[2] for b in boxes); ymax=min(b[3] for b in boxes)
    return None if xmin>=xmax or ymin>=ymax else (xmin,ymin,xmax,ymax)

def _box_area(box): return (box[2]-box[0])*(box[3]-box[1])

def verify_and_clip_overlap(c2017, c2018, label, config=None):
    min_frac = _cfg(config, 'overlap_min_fraction')
    buffer_m = _cfg(config, 'overlap_buffer_m')
    boxes = {'2017': _bb_xy(c2017), '2018': _bb_xy(c2018)}
    areas = {k: _box_area(v) for k,v in boxes.items()}
    inter = _intersect_xy(list(boxes.values()))
    if inter is None:
        msg = f'[OVERLAP] {label}: NO XY overlap — skipping grid.'
        print(msg); raise ValueError(msg)
    inter_area   = _box_area(inter)
    overlap_frac = inter_area / min(areas.values())
    report = {'intersection_area_m2': inter_area, 'cloud_areas': dict(areas),
              'overlap_fraction': overlap_frac, 'was_clipped': False, 'buffer_used': 0.0}
    if overlap_frac >= min_frac:
        print(f'[OVERLAP] {label}: {overlap_frac:.1%} overlap — no clip needed.')
        return c2017, c2018, report
    print(f'[OVERLAP] {label}: {overlap_frac:.1%} — clipping to intersection + {buffer_m}m.')
    z_min = min(c2017.getOwnBB().minCorner()[2], c2018.getOwnBB().minCorner()[2])
    z_max = max(c2017.getOwnBB().maxCorner()[2], c2018.getOwnBB().maxCorner()[2])
    clip_box = cc.ccBBox((inter[0]-buffer_m, inter[1]-buffer_m, z_min-1.0),
                         (inter[2]+buffer_m, inter[3]+buffer_m, z_max+1.0), True)
    def _crop(cloud, orig):
        cropped = cc.cropPointCloud(cloud, clip_box, inside=True)
        return orig if (cropped is None or cropped.size()==0) else cropped
    report['was_clipped']=True; report['buffer_used']=buffer_m
    return _crop(c2017, c2017), _crop(c2018, c2018), report

def _dynamic_octree_level(cloud):
    bb=cloud.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
    diag=sqrt((hi[0]-lo[0])**2+(hi[1]-lo[1])**2+(hi[2]-lo[2])**2)
    return max(6, min(12, int(log2(max(diag/2.0,1.0))))) if diag>0 else 8

def extract_damage_components(source_cloud, c2c_array, sf_name, grid_id, config=None):
    threshold = _cfg(config, 'damage_threshold')
    min_pts   = _cfg(config, 'min_component_pts')
    max_comps = _cfg(config, 'max_components')
    n_above   = int(np.sum(c2c_array > threshold))
    if n_above == 0:
        print(f'[COMP] {grid_id}: no points exceed {threshold}m — skipping.'); return [], [], 0
    print(f'[COMP] {grid_id}: {n_above:,}/{c2c_array.size:,} pts ({100.0*n_above/c2c_array.size:.1f}%) above {threshold}m.')
    sf_dic = source_cloud.getScalarFieldDic()
    if sf_name not in sf_dic:
        print(f'[COMP] {grid_id}: WARNING — SF {sf_name!r} not found.'); return [], [], 0
    source_cloud.setCurrentDisplayedScalarField(sf_dic[sf_name])
    damaged_cloud = source_cloud.filterPointsByScalarValue(threshold, float(np.max(c2c_array))+1.0)
    if damaged_cloud is None or damaged_cloud.size()==0:
        print(f'[COMP] {grid_id}: WARNING — filter returned empty cloud.'); return [], [], 0
    print(f'[COMP] {grid_id}: damaged_cloud has {damaged_cloud.size():,} pts.')
    octree_level = _dynamic_octree_level(damaged_cloud)
    print(f'[COMP] {grid_id}: octree level = {octree_level}.')
    n_proc, components, residuals = cc.ExtractConnectedComponents(
        clouds=[damaged_cloud], octreeLevel=octree_level,
        minComponentSize=min_pts, maxNumberComponents=max_comps, randomColors=False)
    print(f'[COMP] {grid_id}: {len(components)} components, {len(residuals)} residual clouds.')
    cc.deleteEntity(damaged_cloud)
    return components, residuals, octree_level

def _sf_array_from_cloud(cloud):
    sf_dict = cloud.getScalarFieldDic()
    if not sf_dict: return None
    sf = cloud.getScalarField(next(iter(sf_dict)))
    return sf.toNpArrayCopy() if sf else None

def compute_component_stats(components, residuals, grid_id, point_density_per_m2=None):
    stats_list = []
    for i, comp in enumerate(components):
        n_pts = int(comp.size())
        bb=comp.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
        bbox_area = (hi[0]-lo[0])*(hi[1]-lo[1])
        sf_arr = _sf_array_from_cloud(comp)
        if sf_arr is not None and sf_arr.size>0:
            mean_c2c=float(np.mean(sf_arr)); median_c2c=float(np.median(sf_arr))
            std_c2c=float(np.std(sf_arr));   max_c2c=float(np.max(sf_arr))
        else:
            mean_c2c=median_c2c=std_c2c=max_c2c=float('nan')
        rec = {'grid_id':grid_id,'component_id':i,'n_points':n_pts,
               'bbox_area_m2':bbox_area,'bbox_area_sqrt_m':sqrt(max(bbox_area,0.0)),
               'mean_c2c_m':mean_c2c,'median_c2c_m':median_c2c,'std_c2c_m':std_c2c,'max_c2c_m':max_c2c,
               'centroid_x':0.5*(lo[0]+hi[0]),'centroid_y':0.5*(lo[1]+hi[1])}
        if point_density_per_m2 and point_density_per_m2>0:
            rec['estimated_area_m2'] = n_pts/point_density_per_m2
        stats_list.append(rec)
    print(f'[COMP] {grid_id}: stats for {len(stats_list)} components ({sum(int(r.size()) for r in residuals)} residual pts).')
    return stats_list

def _delete_cloud_list(clouds, label):
    for c in clouds:
        try: cc.deleteEntity(c)
        except Exception as e: print(f'[COMP] WARNING — delete failed ({label}): {e}')
    clouds.clear()

print('Connected component functions ready.')

## 7. Power Law Analysis

In [ ]:
def fit_power_law(areas, label='', min_area_m2=10.0):
    areas  = np.asarray(areas, dtype=np.float64)
    fitted = areas[areas >= min_area_m2]
    result = {'alpha_mle':float('nan'),'alpha_ols':float('nan'),'xmin':min_area_m2,
               'n_fitted':0,'ks_stat':float('nan'),'ks_pvalue':float('nan'),
               'r2_loglog':float('nan'),'ll_powerlaw':float('nan'),'ll_lognormal':float('nan'),
               'preferred_distribution':'undetermined'}
    if fitted.size < 5:
        print(f'[POWERLAW] {label}: only {fitted.size} areas >= {min_area_m2}m² — too few.'); return result
    n=fitted.size; xmin=min_area_m2; result['n_fitted']=n
    sum_log = float(np.sum(np.log(fitted/xmin)))
    alpha_mle = (1.0 + n/sum_log) if sum_log>0 else float('nan')
    result['alpha_mle'] = alpha_mle
    n_bins=max(10,int(sqrt(n))); log_edges=np.linspace(np.log10(xmin),np.log10(fitted.max()),n_bins+1)
    counts,_=np.histogram(np.log10(fitted),bins=log_edges); pos=counts>0
    if pos.sum()>=2:
        slope,_,r,_,_=sp_stats.linregress(0.5*(log_edges[:-1]+log_edges[1:])[pos],np.log10(counts[pos].astype(float)))
        result['alpha_ols']=-slope; result['r2_loglog']=r**2
    if np.isfinite(alpha_mle) and alpha_mle>1.0:
        ks=sp_stats.kstest(fitted,'pareto',args=(alpha_mle-1.0,),N=n)
        result['ks_stat']=float(ks.statistic); result['ks_pvalue']=float(ks.pvalue)
        ll_pl=float(n*np.log(alpha_mle-1.0)-n*np.log(xmin)-alpha_mle*np.sum(np.log(fitted/xmin)))
        result['ll_powerlaw']=ll_pl
    else: ll_pl=-np.inf
    mu_ln=float(np.mean(np.log(fitted))); sig_ln=float(np.std(np.log(fitted),ddof=1)) if n>1 else 1.0
    ll_ln=float(np.sum(sp_stats.lognorm.logpdf(fitted,s=sig_ln,scale=np.exp(mu_ln)))) if sig_ln>0 else -np.inf
    result['ll_lognormal']=ll_ln
    if np.isfinite(ll_pl) and np.isfinite(ll_ln):
        result['preferred_distribution']='power_law' if ll_pl>ll_ln else 'lognormal'
    print(f"[POWERLAW] {label}: alpha_MLE={alpha_mle:.3f}, alpha_OLS={result['alpha_ols']:.3f}, "
          f"n={n}, KS={result['ks_stat']:.4f}, preferred={result['preferred_distribution']}")
    return result

def gap_size_distribution_figure(component_stats, grid_id, out_dir):
    out_dir=Path(out_dir)
    areas    = np.array([s['bbox_area_m2'] for s in component_stats], dtype=np.float64)
    mean_c2c = np.array([s['mean_c2c_m']  for s in component_stats], dtype=np.float64)
    fit = fit_power_law(areas, label=grid_id)
    if areas.size==0: return fit
    fig,axes=plt.subplots(2,2,figsize=(12,10))
    fig.suptitle(f'Gap Size Distribution — {grid_id} (2017→2018)',fontsize=13)
    pos_areas=areas[areas>0]
    ax=axes[0,0]
    if pos_areas.size>0:
        log_bins=np.logspace(np.log10(pos_areas.min()),np.log10(pos_areas.max()),max(10,int(sqrt(pos_areas.size))))
        ax.hist(pos_areas,bins=log_bins,edgecolor='k',alpha=0.7,color='red',label='data')
        alpha=fit['alpha_mle']; xmin=fit['xmin']
        if np.isfinite(alpha) and alpha>1.0:
            x_l=np.logspace(np.log10(xmin),np.log10(pos_areas.max()),200)
            pdf=((alpha-1.0)/xmin)*(x_l/xmin)**(-alpha)
            ax.plot(x_l,pdf*np.sum(pos_areas>=xmin)*float(np.mean(np.diff(log_bins))),'k-',lw=2,label=f'PL α={alpha:.2f}')
    ax.set_xscale('log');ax.set_yscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('Count')
    ax.set_title('(a) Log-log histogram');ax.legend(fontsize=8)
    alpha=fit['alpha_mle']; xmin=fit['xmin']; r2=fit.get('r2_loglog',float('nan')); n=fit['n_fitted']
    ax.text(0.97,0.97,f'α={alpha:.2f}\nR²={r2:.3f}\nn={n}\nxmin={xmin:.0f}m²',
            transform=ax.transAxes,fontsize=8,va='top',ha='right',
            bbox=dict(boxstyle='round,pad=0.3',fc='white',alpha=0.8))
    ax=axes[0,1]; sv=np.sort(areas); ecdf=np.arange(1,len(sv)+1)/len(sv)
    ax.step(sv,ecdf,where='post',label='Empirical',lw=1.5,color='red')
    if np.isfinite(fit['alpha_mle']) and fit['alpha_mle']>1.0 and xmin>0:
        x_th=np.sort(sv[sv>=xmin])
        if x_th.size>0:
            tcdf=1.0-(x_th/xmin)**(-(fit['alpha_mle']-1.0))
            fb=np.searchsorted(sv,xmin)/len(sv)
            ax.plot(x_th,fb+(1.0-fb)*tcdf,'k--',lw=1.5,label='Power-law CDF')
    ax.set_xscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('CDF')
    ax.set_title('(b) CDF');ax.legend(fontsize=8)
    ax=axes[1,0]; valid=np.isfinite(mean_c2c)&(areas>0)
    if valid.any(): ax.scatter(areas[valid],mean_c2c[valid],s=8,alpha=0.5,color='red')
    ax.set_xscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('Mean C2C (m)')
    ax.set_title('(c) Gap area vs C2C displacement')
    ax=axes[1,1]; ranks=np.arange(1,len(sv)+1)
    ax.scatter(ranks,sv[::-1],s=8,alpha=0.6,color='red')
    ax.set_xscale('log');ax.set_yscale('log');ax.set_xlabel('Rank');ax.set_ylabel('Gap area (m²)')
    ax.set_title('(d) Rank-size (Zipf)')
    fig.tight_layout(rect=[0,0,1,0.95])
    fname=out_dir/f'{grid_id}_gap_distribution.png'; fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'[DIST] {grid_id}: saved → {fname}')
    return fit

print('Power law and distribution functions ready.')

## 8. Determine Global Coordinate Shift

In [ ]:
files_2017 = sorted(DIR_2017.glob('*.las*'))
print(f'Found {len(files_2017)} files in 2017 directory')

global_shift = None
for probe_file in files_2017:
    try: grid_id_from_name(probe_file)
    except ValueError: continue
    p_probe = gunzip_if_needed(probe_file)
    print(f'[SHIFT] Probing {p_probe.name} ...')
    probe_cloud = cc.loadPointCloud(str(p_probe))
    if probe_cloud is None: continue
    global_shift = tuple(float(x) for x in probe_cloud.getGlobalShift()) if probe_cloud.isShifted() else (0.0,0.0,0.0)
    bb = probe_cloud.getOwnBB()
    print(f'[SHIFT] shift={global_shift}')
    print(f'[SHIFT] BB: X=[{float(bb.minCorner()[0]):.1f},{float(bb.maxCorner()[0]):.1f}] '
          f'Y=[{float(bb.minCorner()[1]):.1f},{float(bb.maxCorner()[1]):.1f}] '
          f'Z=[{float(bb.minCorner()[2]):.1f},{float(bb.maxCorner()[2]):.1f}]')
    safe_delete(probe_cloud)
    break

if global_shift is None: raise RuntimeError('Could not determine shift from any 2017 file')
print(f'\nUsing global shift {global_shift} for ALL files.')

## 9. Main Processing Loop
Each grid tile is wrapped in a `GRID_TIMEOUT_SEC` timeout. If cloudComPy hangs on a file, it is skipped and logged — the loop continues with the next grid.

In [ ]:
all_summaries  = []
all_thresholds = []
all_extreme    = []
all_norm       = []
all_cc_stats   = []
all_pl_fits    = []
skipped_grids  = []
n_processed = n_skip_2018 = 0
norm_done = False

for idx, f2017 in enumerate(files_2017, start=1):
    loop_t0 = tnow()
    try: grid = grid_id_from_name(f2017)
    except ValueError: continue

    f2018 = find_match(DIR_2018, grid)
    if f2018 is None:
        print(f'  [SKIP] {grid}: no 2018 match'); n_skip_2018+=1; continue

    print(f'\n{"="*70}\n  [{idx}/{len(files_2017)}]  GRID {grid.upper()}\n{"="*70}')

    # Locals to clean up on timeout or error
    c2017=c2018=c2017s=c2018s=None

    try:
        start_timeout(GRID_TIMEOUT_SEC)

        p2017 = gunzip_if_needed(f2017)
        p2018 = gunzip_if_needed(f2018)
        c2017, _ = load_cloud(p2017, shift=global_shift)
        c2018, _ = load_cloud(p2018, shift=global_shift)

        # BB sanity check
        br=c2017.getOwnBB(); bo=c2018.getOwnBB()
        dist=(abs(float(br.minCorner()[0])-float(bo.minCorner()[0]))+
              abs(float(br.minCorner()[1])-float(bo.minCorner()[1])))
        print(f'  [CHECK] 2017 vs 2018 BB offset: {dist:.1f}m ({"OK" if dist<=100 else "WARN — possible shift mismatch"})')

        # Normalisation check
        if not norm_done:
            for c,lbl in [(c2017,f'{grid}_2017'),(c2018,f'{grid}_2018')]:
                info=check_normalization(c,lbl); print_norm_report(info); all_norm.append(info)
            norm_done=True
        else:
            info=check_normalization(c2017,f'{grid}_2017'); all_norm.append(info)

        # Subsample
        did_sub=False
        if USE_SUBSAMPLE:
            c2017s=subsample_cloud(c2017,SUBSAMPLE_N); c2018s=subsample_cloud(c2018,SUBSAMPLE_N)
            did_sub=(c2017s is not c2017)
        else:
            c2017s=c2017; c2018s=c2018

        # Overlap check & clip
        c2017s, c2018s, overlap_report = verify_and_clip_overlap(c2017s, c2018s, grid, CC_CONFIG)

        # C2C 2017 → 2018
        sf_name  = compute_c2c(c2017s, c2018s, 'C2C_2017_2018', threads=THREADS)
        c2c_arr  = get_sf_as_array(c2017s, sf_name)
        cc.SavePointCloud(c2017s, str(BIN_DIR/f'{grid}_2017_c2c.bin'))
        print(f'  [DATA] {len(c2c_arr):,} pts extracted')

        # Stats
        s1 = c2c_summary_stats(c2c_arr, f'{grid}_2017to2018')
        s1['grid']=grid; all_summaries.append(s1)
        tf = c2c_threshold_fractions(c2c_arr)
        row={'grid':grid}; [row.update({f'gte_{t}m':tf[t]}) for t in C2C_THRESHOLDS]
        all_thresholds.append(row)
        ev=extreme_value_report(c2c_arr,f'{grid}_2017to2018'); ev['grid']=grid; all_extreme.append(ev)
        plot_grid(grid, c2c_arr)

        # Connected components
        density = c2017s.size() / max(_box_area(_bb_xy(c2017s)), 1.0)
        comps, resids, oct_lvl = extract_damage_components(c2017s, c2c_arr, sf_name, grid, CC_CONFIG)
        comp_stats = compute_component_stats(comps, resids, grid, density)
        pl_fit = gap_size_distribution_figure(comp_stats, grid, PLOT_DIR)
        _delete_cloud_list(comps, f'{grid} components')
        _delete_cloud_list(resids, f'{grid} residuals')

        for row in comp_stats: all_cc_stats.append(row)
        all_pl_fits.append({'grid':grid,'alpha_mle':pl_fit['alpha_mle'],
                            'alpha_ols':pl_fit['alpha_ols'],'n_fitted':pl_fit['n_fitted'],
                            'ks_stat':pl_fit['ks_stat'],'ks_pvalue':pl_fit['ks_pvalue'],
                            'r2_loglog':pl_fit['r2_loglog'],'preferred':pl_fit['preferred_distribution'],
                            'overlap_fraction':overlap_report['overlap_fraction'],
                            'was_clipped':overlap_report['was_clipped']})

        cancel_timeout()
        n_processed += 1
        print(f'\n  [DONE] {grid.upper()} in {fmt_s(tnow()-loop_t0)} | mem={mem_gb():.2f}GB')

    except GridTimeoutError:
        cancel_timeout()
        print(f'  [TIMEOUT] {grid.upper()} exceeded {GRID_TIMEOUT_SEC}s — skipping.')
        skipped_grids.append({'grid':grid,'reason':'timeout','elapsed_s':tnow()-loop_t0})

    except Exception as e:
        cancel_timeout()
        print(f'  [ERROR] {grid.upper()}: {e} — skipping.')
        skipped_grids.append({'grid':grid,'reason':str(e),'elapsed_s':tnow()-loop_t0})

    finally:
        cancel_timeout()
        for obj in [c2017s, c2018s, c2017, c2018]:
            if obj is not None: safe_delete(obj)

print(f'\n{"="*70}')
print(f'  Processed : {n_processed}')
print(f'  No 2018   : {n_skip_2018}')
print(f'  Skipped   : {len(skipped_grids)}')
if skipped_grids:
    for s in skipped_grids:
        print(f'    {s["grid"]} — {s["reason"]} ({s["elapsed_s"]:.0f}s)')
print(f'{"="*70}')

## 10. Write CSV Outputs

In [ ]:
write_csv(CSV_DIR/'c2c_summary_stats.csv',       all_summaries)
write_csv(CSV_DIR/'c2c_threshold_fractions.csv',  all_thresholds)
write_csv(CSV_DIR/'c2c_extreme_values.csv',        all_extreme)
write_csv(CSV_DIR/'normalisation_checks.csv',      all_norm)
write_csv(CSV_DIR/'component_stats.csv',           all_cc_stats)
write_csv(CSV_DIR/'power_law_fits.csv',            all_pl_fits)
write_csv(CSV_DIR/'skipped_grids.csv',             skipped_grids)
print('All CSVs written.')

## 11. Results Review

In [ ]:
import pandas as pd
print('=== C2C Summary (2017→2018) ===')
print(pd.read_csv(CSV_DIR/'c2c_summary_stats.csv')[['grid','n_clean','mean','median','p95']].to_string(index=False))
print('\n=== Power Law Fits ===')
print(pd.read_csv(CSV_DIR/'power_law_fits.csv')[['grid','alpha_mle','alpha_ols','n_fitted','ks_pvalue','preferred']].to_string(index=False))
if (CSV_DIR/'skipped_grids.csv').exists():
    sk=pd.read_csv(CSV_DIR/'skipped_grids.csv')
    if len(sk)>0:
        print('\n=== Skipped Grids ===')
        print(sk.to_string(index=False))